# Demographics Summary

Descriptive statistics (diagnosis, age, sex, scanning site) for the full analysis sample —
everyone pooled together, not split by diagnostic group.

Sample is defined the same way as `2ai-roi-comparison.ipynb`: every subject-specific habenula
ROI in `binary_rois/`, minus the two known excluded subjects, giving **N = 1,480**.

In [1]:
import os
import os.path as op
import re
import glob

import numpy as np
import pandas as pd

## 1. Define Paths and Sample

Same exclusion logic as `2ai-roi-comparison.ipynb`: `sub-0050353` and `sub-0050369` are present
in `binary_rois/` but excluded from the confirmed N=1,480 analysis sample.

In [2]:
data_dir = "./dset"

# Subject-specific (hand-drawn) binary ROIs - one file per participant included in the sample
binary_rois_dir = op.join(data_dir, "seed-regions", "subj-spec-hbs", "binary_rois")

# Full ABIDE phenotypic file
participants_tsv = op.join(data_dir, "participants.tsv")

# Where to save the demographics summary
output_dir = op.join(data_dir, "derivatives", "demographics")
os.makedirs(output_dir, exist_ok=True)
output_csv = op.join(output_dir, "demographics_summary.csv")

# Known mismatched subjects excluded from the confirmed N=1,480 sample (see 2ai-roi-comparison.ipynb)
EXCLUDED_SUBJECTS = {"sub-0050353", "sub-0050369"}

In [3]:
# Find every subject-specific binary ROI file and pull out subject IDs
roi_files_all = sorted(glob.glob(op.join(binary_rois_dir, "*.nii.gz")))
sub_id_pattern = re.compile(r"(sub-\d+)")

def get_sub_id(f):
    return sub_id_pattern.search(op.basename(f)).group(1)

subject_ids_all = [get_sub_id(f) for f in roi_files_all]

# Drop the known excluded subjects
included_subjects = sorted(set(subject_ids_all) - EXCLUDED_SUBJECTS)

print(f"Found {len(roi_files_all)} subject-specific habenula ROIs")
print(f"Excluding {len(set(subject_ids_all)) - len(included_subjects)} known excluded subjects: {sorted(EXCLUDED_SUBJECTS)}")
print(f"\n==> N = {len(included_subjects)} participants in the demographics sample")

Found 1482 subject-specific habenula ROIs
Excluding 2 known excluded subjects: ['sub-0050353', 'sub-0050369']

==> N = 1480 participants in the demographics sample


## 2. Load `participants.tsv` and Filter to the Included Sample

`dset/participants.tsv` is the full ABIDE phenotypic file (all sites, all subjects that were
ever considered). We filter it down to just the 1,480 participant IDs in our sample.

In [4]:
participants_df = pd.read_csv(participants_tsv, sep="\t", low_memory=False)

demo_df = participants_df[participants_df["participant_id"].isin(included_subjects)].copy()

# Sanity check: every included subject should have a matching row
missing = sorted(set(included_subjects) - set(demo_df["participant_id"]))
print(f"{len(demo_df)} of {len(included_subjects)} included participants found in participants.tsv")
if missing:
    print(f"WARNING - missing from participants.tsv: {missing}")
else:
    print("All included participants matched - no missing rows.")

1480 of 1480 included participants found in participants.tsv
All included participants matched - no missing rows.


## 3. Diagnosis (ASD vs. NT)

ABIDE coding: `DX_GROUP` 1 = Autism (ASD), 2 = Control (NT).

In [5]:
dx_map = {1: "ASD", 2: "NT"}
demo_df["diagnosis"] = demo_df["DX_GROUP"].map(dx_map)

dx_counts = demo_df["diagnosis"].value_counts(dropna=False)
dx_pct = demo_df["diagnosis"].value_counts(normalize=True, dropna=False) * 100

print("Diagnosis counts (N total = {}):".format(len(demo_df)))
for label in ["ASD", "NT"]:
    n = dx_counts.get(label, 0)
    pct = dx_pct.get(label, 0.0)
    print(f"  {label}: {n} ({pct:.1f}%)")

n_unmapped = demo_df["diagnosis"].isna().sum()
if n_unmapped:
    print(f"  Unmapped/missing DX_GROUP: {n_unmapped}")

Diagnosis counts (N total = 1480):
  ASD: 661 (44.7%)
  NT: 819 (55.3%)


## 4. Age at Scan

In [6]:
age = demo_df["AGE_AT_SCAN"]

print(f"Age at scan (N with data = {age.notna().sum()} of {len(demo_df)}):")
print(f"  Mean (SD):   {age.mean():.2f} ({age.std():.2f})")
print(f"  Median:      {age.median():.2f}")
print(f"  Range:       [{age.min():.2f}, {age.max():.2f}]")
if age.isna().sum():
    print(f"  Missing:     {age.isna().sum()}")

Age at scan (N with data = 1480 of 1480):
  Mean (SD):   16.49 (8.15)
  Median:      14.00
  Range:       [5.22, 64.00]


## 5. Sex

ABIDE coding: `SEX` 1 = Male, 2 = Female.

In [7]:
sex_map = {1: "Male", 2: "Female"}
demo_df["sex_label"] = demo_df["SEX"].map(sex_map)

sex_counts = demo_df["sex_label"].value_counts(dropna=False)
sex_pct = demo_df["sex_label"].value_counts(normalize=True, dropna=False) * 100

print(f"Sex (N total = {len(demo_df)}):")
for label in ["Male", "Female"]:
    n = sex_counts.get(label, 0)
    pct = sex_pct.get(label, 0.0)
    print(f"  {label}: {n} ({pct:.1f}%)")

n_unmapped = demo_df["sex_label"].isna().sum()
if n_unmapped:
    print(f"  Unmapped/missing SEX: {n_unmapped}")

Sex (N total = 1480):
  Male: 1185 (80.1%)
  Female: 295 (19.9%)


## 6. Scanning Site

In [8]:
site_counts = demo_df["SITE_ID"].value_counts(dropna=False).sort_index()

print(f"Scanning site (N sites = {demo_df['SITE_ID'].nunique()}, N total = {len(demo_df)}):\n")
for site, n in site_counts.items():
    pct = 100 * n / len(demo_df)
    print(f"  {site}: {n} ({pct:.1f}%)")

Scanning site (N sites = 34, N total = 1480):

  BNI_1: 25 (1.7%)
  CALTECH: 14 (0.9%)
  CMU: 11 (0.7%)
  EMC_1: 3 (0.2%)
  ETH_1: 32 (2.2%)
  GU_1: 57 (3.9%)
  IP_1: 38 (2.6%)
  IU_1: 39 (2.6%)
  KKI: 24 (1.6%)
  KKI_1: 164 (11.1%)
  KUL_3: 28 (1.9%)
  LEUVEN_1: 26 (1.8%)
  LEUVEN_2: 26 (1.8%)
  MAX_MUN: 39 (2.6%)
  NYU: 163 (11.0%)
  NYU_1: 73 (4.9%)
  OHSU: 10 (0.7%)
  OHSU_1: 73 (4.9%)
  OILH_2: 46 (3.1%)
  OLIN: 24 (1.6%)
  PITT: 42 (2.8%)
  SBL: 26 (1.8%)
  SDSU: 26 (1.8%)
  SDSU_1: 56 (3.8%)
  STANFORD: 23 (1.6%)
  TCD_1: 32 (2.2%)
  TRINITY: 42 (2.8%)
  UCLA_1: 62 (4.2%)
  UCLA_2: 14 (0.9%)
  UM_1: 78 (5.3%)
  UM_2: 33 (2.2%)
  USM: 63 (4.3%)
  USM_1: 30 (2.0%)
  YALE: 38 (2.6%)


## 7. Breakdown by Diagnostic Group (ASD vs. NT)

Everything above is pooled across the full N=1,480 sample. This section reports the same
diagnosis/age/sex/site breakdown separately for the ASD and NT groups, for reference alongside
the pooled numbers - it doesn't replace them.

In [9]:
group_rows = []

all_sites = sorted(demo_df["SITE_ID"].dropna().unique())

for label in ["ASD", "NT"]:
    g = demo_df[demo_df["diagnosis"] == label]
    g_age = g["AGE_AT_SCAN"]
    g_sex = g["sex_label"].value_counts()
    g_site = g["SITE_ID"].value_counts().reindex(all_sites, fill_value=0)

    group_rows.append({"category": "N", "measure": "N", "group": label, "value": len(g)})
    group_rows.append({"category": "Age (years)", "measure": "Mean (SD)", "group": label,
                        "value": f"{g_age.mean():.2f} ({g_age.std():.2f})"})
    group_rows.append({"category": "Age (years)", "measure": "Median", "group": label,
                        "value": f"{g_age.median():.2f}"})
    group_rows.append({"category": "Age (years)", "measure": "Range", "group": label,
                        "value": f"{g_age.min():.2f} - {g_age.max():.2f}"})
    for sex_label in ["Male", "Female"]:
        n = g_sex.get(sex_label, 0)
        pct = 100 * n / len(g) if len(g) else 0.0
        group_rows.append({"category": "Sex", "measure": sex_label, "group": label,
                            "value": f"{n} ({pct:.1f}%)"})
    for site, n in g_site.items():
        pct = 100 * n / len(g) if len(g) else 0.0
        group_rows.append({"category": "Scanning Site", "measure": site, "group": label,
                            "value": f"{n} ({pct:.1f}%)"})

group_long_df = pd.DataFrame(group_rows)
group_by_df = group_long_df.pivot_table(index=["category", "measure"], columns="group",
                                         values="value", aggfunc="first").reindex(
    columns=["ASD", "NT"])
# Preserve a sensible row order (category order of first appearance, sites sorted within category)
row_order = list(dict.fromkeys(zip(group_long_df["category"], group_long_df["measure"])))
group_by_df = group_by_df.loc[row_order]

group_by_csv = op.join(output_dir, "demographics_by_group.csv")
group_by_df.to_csv(group_by_csv)
print(f"Breakdown by diagnostic group saved to: {group_by_csv}\n")

group_by_df

Breakdown by diagnostic group saved to: ./dset/derivatives/demographics/demographics_by_group.csv



group                             ASD            NT
category      measure                              
N             N                   661           819
Age (years)   Mean (SD)  16.68 (8.23)  16.34 (8.08)
              Median            14.27         13.77
              Range      5.22 - 59.00  5.89 - 64.00
Sex           Male        572 (86.5%)   613 (74.8%)
              Female       89 (13.5%)   206 (25.2%)
Scanning Site BNI_1         13 (2.0%)     12 (1.5%)
              CALTECH        4 (0.6%)     10 (1.2%)
              CMU            6 (0.9%)      5 (0.6%)
              EMC_1          3 (0.5%)      0 (0.0%)
              ETH_1          9 (1.4%)     23 (2.8%)
              GU_1          26 (3.9%)     31 (3.8%)
              IP_1          15 (2.3%)     23 (2.8%)
              IU_1          19 (2.9%)     20 (2.4%)
              KKI            6 (0.9%)     18 (2.2%)
              KKI_1         39 (5.9%)   125 (15.3%)
              KUL_3         28 (4.2%)      0 (0.0%)
              LEUVEN_1      12 (1.8%)     14 (1.7%)
              LEUVEN_2      10 (1.5%)     16 (2.0%)
              MAX_MUN       15 (2.3%)     24 (2.9%)
              NYU          68 (10.3%)    95 (11.6%)
              NYU_1         43 (6.5%)     30 (3.7%)
              OHSU           6 (0.9%)      4 (0.5%)
              OHSU_1        34 (5.1%)     39 (4.8%)
              OILH_2        16 (2.4%)     30 (3.7%)
              OLIN          12 (1.8%)     12 (1.5%)
              PITT          20 (3.0%)     22 (2.7%)
              SBL           12 (1.8%)     14 (1.7%)
              SDSU           8 (1.2%)     18 (2.2%)
              SDSU_1        32 (4.8%)     24 (2.9%)
              STANFORD      11 (1.7%)     12 (1.5%)
              TCD_1         13 (2.0%)     19 (2.3%)
              TRINITY       19 (2.9%)     23 (2.8%)
              UCLA_1        33 (5.0%)     29 (3.5%)
              UCLA_2         8 (1.2%)      6 (0.7%)
              UM_1          32 (4.8%)     46 (5.6%)
              UM_2          13 (2.0%)     20 (2.4%)
              USM           40 (6.1%)     23 (2.8%)
              USM_1         15 (2.3%)     15 (1.8%)
              YALE          21 (3.2%)     17 (2.1%)

## 8. Group Comparisons (ASD vs. NT)

The tables above are pooled across everyone. It's still standard practice to check that the
two diagnostic groups are reasonably well-matched on age, sex, and site, so this section runs
the group-comparison tests typically reported in a Table 1 (independent-samples t-test for age,
chi-square tests of independence for sex and site) - it does not change the pooled counts above.

In [10]:
from scipy import stats

asd_age = demo_df.loc[demo_df["diagnosis"] == "ASD", "AGE_AT_SCAN"].dropna()
nt_age = demo_df.loc[demo_df["diagnosis"] == "NT", "AGE_AT_SCAN"].dropna()

# Levene's test to check the equal-variance assumption, then use Welch's t-test if it's violated
levene_stat, levene_p = stats.levene(asd_age, nt_age)
equal_var = levene_p >= 0.05

age_t, age_p = stats.ttest_ind(asd_age, nt_age, equal_var=equal_var)
age_df = (len(asd_age) + len(nt_age) - 2) if equal_var else None

print(f"Levene's test for equal variances: F = {levene_stat:.3f}, p = {levene_p:.4f}"
      f" -> {'equal' if equal_var else 'unequal'} variances assumed")
print(f"\nAge, ASD (M={asd_age.mean():.2f}, SD={asd_age.std():.2f}, N={len(asd_age)}) vs."
      f" NT (M={nt_age.mean():.2f}, SD={nt_age.std():.2f}, N={len(nt_age)}):")
if equal_var:
    print(f"  Student's t({age_df}) = {age_t:.3f}, p = {age_p:.4f}")
else:
    print(f"  Welch's t = {age_t:.3f}, p = {age_p:.4f}")

Levene's test for equal variances: F = 0.062, p = 0.8041 -> equal variances assumed

Age, ASD (M=16.68, SD=8.23, N=661) vs. NT (M=16.34, SD=8.08, N=819):
  Student's t(1478) = 0.816, p = 0.4149


In [11]:
sex_table = pd.crosstab(demo_df["diagnosis"], demo_df["sex_label"])
sex_chi2, sex_chi2_p, sex_dof, sex_expected = stats.chi2_contingency(sex_table)

print("Diagnosis x Sex contingency table:")
print(sex_table)
print(f"\nChi-square test of independence: chi2({sex_dof}) = {sex_chi2:.3f}, p = {sex_chi2_p:.4f}")
if (sex_expected < 5).any():
    print("  Note: at least one expected cell count is < 5 - chi-square approximation may be unreliable.")

Diagnosis x Sex contingency table:
sex_label  Female  Male
diagnosis              
ASD            89   572
NT            206   613

Chi-square test of independence: chi2(1) = 30.583, p = 0.0000


In [12]:
site_table = pd.crosstab(demo_df["diagnosis"], demo_df["SITE_ID"])
site_chi2, site_chi2_p, site_dof, site_expected = stats.chi2_contingency(site_table)

print(f"Diagnosis x Site contingency table: {site_table.shape[0]} diagnoses x {site_table.shape[1]} sites")
print(f"\nChi-square test of independence: chi2({site_dof}) = {site_chi2:.3f}, p = {site_chi2_p:.4f}")
n_low_expected = (site_expected < 5).sum()
if n_low_expected:
    pct_low = 100 * n_low_expected / site_expected.size
    print(f"  Note: {n_low_expected}/{site_expected.size} expected cell counts ({pct_low:.0f}%) are < 5"
          " - with this many sites, the chi-square approximation may be unreliable; interpret with caution.")

Diagnosis x Site contingency table: 2 diagnoses x 34 sites

Chi-square test of independence: chi2(33) = 109.194, p = 0.0000
  Note: 4/68 expected cell counts (6%) are < 5 - with this many sites, the chi-square approximation may be unreliable; interpret with caution.


## 9. Summary Table

One combined table (everyone pooled, not split by diagnostic group), plus the ASD-vs-NT
group-comparison statistics from Section 8, saved to
`dset/derivatives/demographics/demographics_summary.csv`. The by-group breakdown from
Section 7 is saved separately to `dset/derivatives/demographics/demographics_by_group.csv`.

In [13]:
summary_rows = []

summary_rows.append({"category": "Total N", "measure": "N", "value": len(demo_df)})

for label in ["ASD", "NT"]:
    summary_rows.append({"category": "Diagnosis", "measure": label,
                          "value": f"{dx_counts.get(label, 0)} ({dx_pct.get(label, 0.0):.1f}%)"})

summary_rows.append({"category": "Age (years)", "measure": "Mean (SD)",
                      "value": f"{age.mean():.2f} ({age.std():.2f})"})
summary_rows.append({"category": "Age (years)", "measure": "Range",
                      "value": f"{age.min():.2f} - {age.max():.2f}"})

for label in ["Male", "Female"]:
    summary_rows.append({"category": "Sex", "measure": label,
                          "value": f"{sex_counts.get(label, 0)} ({sex_pct.get(label, 0.0):.1f}%)"})

for site, n in site_counts.items():
    pct = 100 * n / len(demo_df)
    summary_rows.append({"category": "Scanning Site", "measure": site,
                          "value": f"{n} ({pct:.1f}%)"})

age_test_label = f"Student's t({age_df})" if equal_var else "Welch's t"
summary_rows.append({"category": "Group Comparison (ASD vs. NT)", "measure": "Age",
                      "value": f"{age_test_label} = {age_t:.3f}, p = {age_p:.4f}"})
summary_rows.append({"category": "Group Comparison (ASD vs. NT)", "measure": "Sex",
                      "value": f"chi2({sex_dof}) = {sex_chi2:.3f}, p = {sex_chi2_p:.4f}"})
summary_rows.append({"category": "Group Comparison (ASD vs. NT)", "measure": "Site",
                      "value": f"chi2({site_dof}) = {site_chi2:.3f}, p = {site_chi2_p:.4f}"})

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(output_csv, index=False)
print(f"Summary table saved to: {output_csv}")

summary_df

Summary table saved to: ./dset/derivatives/demographics/demographics_summary.csv


,category,measure,value
0,Total N,N,1480
1,Diagnosis,ASD,661 (44.7%)
2,Diagnosis,NT,819 (55.3%)
3,Age (years),Mean (SD),16.49 (8.15)
4,Age (years),Range,5.22 - 64.00
5,Sex,Male,1185 (80.1%)
6,Sex,Female,295 (19.9%)
7,Scanning Site,BNI_1,25 (1.7%)
8,Scanning Site,CALTECH,14 (0.9%)
9,Scanning Site,CMU,11 (0.7%)
